# Reproducing & Extending STAtten: Multi-Scale Block Attention

**Paper:** Lee et al., *Spiking Transformer with Spatial-Temporal Attention*, CVPR 2025

**100% self-contained** — no repo cloning, no external train.py, no patching.

## Objectives
1. Reproduce STAtten on CIFAR-10/100
2. Propose **MSB-STAtten** (Multi-Scale Block Attention)
3. Compare SDT vs STAtten vs MSB-STAtten

In [1]:
#@title 1. Install & Verify
!pip install -q spikingjelly==0.0.0.0.12 cupy-cuda12x

import torch
import torch.nn as nn
from spikingjelly.clock_driven.neuron import MultiStepLIFNode
from spikingjelly.clock_driven import functional
print(f"PyTorch {torch.__version__} | CUDA {torch.cuda.is_available()} | {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print("All imports OK.")

2026-03-24 05:57:28.181410: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774331848.205848     230 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774331848.213647     230 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774331848.234444     230 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774331848.234465     230 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774331848.234467     230 computation_placer.cc:177] computation placer alr

PyTorch 2.10.0+cu128 | CUDA True | Tesla T4
All imports OK.


In [2]:
#@title 2. Model Definition (fully embedded — SDT, STAtten, MSB_STAtten)

import torch
import torch.nn as nn
from spikingjelly.clock_driven.neuron import MultiStepLIFNode
from spikingjelly.clock_driven import functional


class MS_SPS(nn.Module):
    """Spiking Patch Splitter: images -> spike embeddings (faithful to original)."""
    def __init__(self, img_size=32, in_channels=3, embed_dims=256, pooling_stat="0011"):
        super().__init__()
        self.pooling_stat = pooling_stat

        self.proj_conv = nn.Conv2d(in_channels, embed_dims//8, 3, 1, 1, bias=False)
        self.proj_bn = nn.BatchNorm2d(embed_dims//8)
        self.proj_lif = MultiStepLIFNode(tau=2.0, detach_reset=True, backend="cupy")
        self.maxpool = nn.MaxPool2d(3, 2, 1)

        self.proj_conv1 = nn.Conv2d(embed_dims//8, embed_dims//4, 3, 1, 1, bias=False)
        self.proj_bn1 = nn.BatchNorm2d(embed_dims//4)
        self.proj_lif1 = MultiStepLIFNode(tau=2.0, detach_reset=True, backend="cupy")
        self.maxpool1 = nn.MaxPool2d(3, 2, 1)

        self.proj_conv2 = nn.Conv2d(embed_dims//4, embed_dims//2, 3, 1, 1, bias=False)
        self.proj_bn2 = nn.BatchNorm2d(embed_dims//2)
        self.proj_lif2 = MultiStepLIFNode(tau=2.0, detach_reset=True, backend="cupy")
        self.maxpool2 = nn.MaxPool2d(3, 2, 1)

        self.proj_conv3 = nn.Conv2d(embed_dims//2, embed_dims, 3, 1, 1, bias=False)
        self.proj_bn3 = nn.BatchNorm2d(embed_dims)
        self.proj_lif3 = MultiStepLIFNode(tau=2.0, detach_reset=True, backend="cupy")
        self.maxpool3 = nn.MaxPool2d(3, 2, 1)

        self.rpe_conv = nn.Conv2d(embed_dims, embed_dims, 3, 1, 1, bias=False)
        self.rpe_bn = nn.BatchNorm2d(embed_dims)

    def forward(self, x):
        T, B, _, H, W = x.shape
        ratio = 1

        # Stage 0
        x = self.proj_conv(x.flatten(0, 1))
        x = self.proj_bn(x).reshape(T, B, -1, H//ratio, W//ratio).contiguous()
        x = self.proj_lif(x).flatten(0, 1)
        if self.pooling_stat[0] == '1':
            x = self.maxpool(x); ratio *= 2

        # Stage 1
        x = self.proj_conv1(x)
        x = self.proj_bn1(x).reshape(T, B, -1, H//ratio, W//ratio).contiguous()
        x = self.proj_lif1(x).flatten(0, 1)
        if self.pooling_stat[1] == '1':
            x = self.maxpool1(x); ratio *= 2

        # Stage 2
        x = self.proj_conv2(x)
        x = self.proj_bn2(x).reshape(T, B, -1, H//ratio, W//ratio).contiguous()
        x = self.proj_lif2(x).flatten(0, 1)
        if self.pooling_stat[2] == '1':
            x = self.maxpool2(x); ratio *= 2

        # Stage 3
        x = self.proj_conv3(x)
        x = self.proj_bn3(x)
        if self.pooling_stat[3] == '1':
            x = self.maxpool3(x); ratio *= 2

        x_feat = x
        x = self.proj_lif3(x.reshape(T, B, -1, H//ratio, W//ratio).contiguous())
        x = self.rpe_bn(self.rpe_conv(x.flatten(0, 1)))
        x = (x + x_feat).reshape(T, B, -1, H//ratio, W//ratio).contiguous()
        return x


class MS_MLP(nn.Module):
    def __init__(self, dim, mlp_ratio=4):
        super().__init__()
        hid = int(dim * mlp_ratio)
        self.fc1 = nn.Conv2d(dim, hid, 1); self.bn1 = nn.BatchNorm2d(hid)
        self.lif1 = MultiStepLIFNode(tau=2.0, detach_reset=True, backend="cupy")
        self.fc2 = nn.Conv2d(hid, dim, 1); self.bn2 = nn.BatchNorm2d(dim)
        self.lif2 = MultiStepLIFNode(tau=2.0, detach_reset=True, backend="cupy")
        self.hid = hid

    def forward(self, x):
        T, B, C, H, W = x.shape
        identity = x
        x = self.lif1(x)
        x = self.bn1(self.fc1(x.flatten(0, 1))).reshape(T, B, self.hid, H, W).contiguous()
        x = self.lif2(x)
        x = self.bn2(self.fc2(x.flatten(0, 1))).reshape(T, B, C, H, W).contiguous()
        return x + identity


class MS_Attention(nn.Module):
    """Unified attention: SDT, STAtten, MSB_STAtten."""
    def __init__(self, dim, num_heads=8, attention_mode="STAtten", chunk_size=2):
        super().__init__()
        self.dim = dim; self.num_heads = num_heads
        self.attention_mode = attention_mode; self.chunk_size = chunk_size
        self.shortcut_lif = MultiStepLIFNode(tau=2.0, detach_reset=True, backend="cupy")
        for name in ['q', 'k', 'v']:
            setattr(self, f'{name}_conv', nn.Conv2d(dim, dim, 1, bias=False))
            setattr(self, f'{name}_bn', nn.BatchNorm2d(dim))
            setattr(self, f'{name}_lif', MultiStepLIFNode(tau=2.0, detach_reset=True, backend="cupy"))
        self.attn_lif = MultiStepLIFNode(tau=2.0, v_threshold=0.5, detach_reset=True, backend="cupy")
        self.proj_conv = nn.Conv2d(dim, dim, 1)
        self.proj_bn = nn.BatchNorm2d(dim)

    def _get_qkv(self, x):
        T, B, C, H, W = x.shape
        N = H * W; hd = C // self.num_heads
        xf = x.flatten(0, 1)
        out = []
        for name in ['q', 'k', 'v']:
            conv = getattr(self, f'{name}_conv')
            bn = getattr(self, f'{name}_bn')
            lif = getattr(self, f'{name}_lif')
            o = lif(bn(conv(xf)).reshape(T, B, C, H, W).contiguous())
            o = o.flatten(3).transpose(-1, -2).reshape(T, B, N, self.num_heads, hd)
            o = o.permute(0, 1, 3, 2, 4).contiguous()
            out.append(o)
        return out[0], out[1], out[2], N, hd

    def _block_attn(self, q, k, v, cs, N, hd, H):
        T, Bb, nh = q.shape[:3]
        sc = 1.0 / H
        nc = T // cs
        q = q.reshape(nc, cs, Bb, nh, N, hd).permute(0, 2, 3, 1, 4, 5).reshape(nc, Bb, nh, cs*N, hd)
        k = k.reshape(nc, cs, Bb, nh, N, hd).permute(0, 2, 3, 1, 4, 5).reshape(nc, Bb, nh, cs*N, hd)
        v = v.reshape(nc, cs, Bb, nh, N, hd).permute(0, 2, 3, 1, 4, 5).reshape(nc, Bb, nh, cs*N, hd)
        a = torch.matmul(k.transpose(-2, -1), v) * sc
        o = torch.matmul(q, a)
        return o.reshape(nc, Bb, nh, cs, N, hd).permute(0, 3, 1, 2, 4, 5).reshape(T, Bb, nh, N, hd)

    def forward(self, x):
        T, B, C, H, W = x.shape
        identity = x
        x = self.shortcut_lif(x)
        q, k, v, N, hd = self._get_qkv(x)

        if self.attention_mode == "STAtten":
            output = self._block_attn(q, k, v, self.chunk_size, N, hd, H)

        elif self.attention_mode == "MSB_STAtten":
            h1 = self.num_heads // 2; h2 = self.num_heads - h1
            o1 = self._block_attn(q[:,:,:h1], k[:,:,:h1], v[:,:,:h1], 2, N, hd, H)
            o2 = self._block_attn(q[:,:,h1:], k[:,:,h1:], v[:,:,h1:], T, N, hd, H)
            output = torch.cat([o1, o2], dim=2)

        elif self.attention_mode == "SDT":
            kv = k.mul(v).sum(dim=-2, keepdim=True)
            output = q.mul(kv)
        else:
            raise ValueError(f"Unknown: {self.attention_mode}")

        x = output.transpose(4, 3).reshape(T, B, C, N).contiguous()
        x = self.attn_lif(x).reshape(T, B, C, H, W)
        x = self.proj_bn(self.proj_conv(x.flatten(0, 1))).reshape(T, B, C, H, W).contiguous()
        return x + identity


class MS_Block(nn.Module):
    def __init__(self, dim, num_heads=8, mlp_ratio=4, attention_mode="STAtten", chunk_size=2):
        super().__init__()
        self.attn = MS_Attention(dim, num_heads, attention_mode, chunk_size)
        self.mlp = MS_MLP(dim, mlp_ratio)
    def forward(self, x):
        return self.mlp(self.attn(x))


class SpikingTransformer(nn.Module):
    def __init__(self, img_size=32, in_channels=3, num_classes=10,
                 embed_dims=512, num_heads=8, depth=2, mlp_ratio=4,
                 T=4, chunk_size=2, pooling_stat="0011", attention_mode="STAtten"):
        super().__init__()
        self.T = T
        self.patch_embed = MS_SPS(img_size, in_channels, embed_dims, pooling_stat)
        self.blocks = nn.ModuleList([
            MS_Block(embed_dims, num_heads, mlp_ratio, attention_mode, chunk_size)
            for _ in range(depth)
        ])
        self.head_lif = MultiStepLIFNode(tau=2.0, detach_reset=True, backend="cupy")
        self.head = nn.Linear(embed_dims, num_classes)
        self.apply(self._init_w)

    def _init_w(self, m):
        if isinstance(m, nn.Conv2d):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.BatchNorm2d):
            nn.init.ones_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, x):
        x = x.unsqueeze(0).repeat(self.T, 1, 1, 1, 1)
        x = self.patch_embed(x)
        for blk in self.blocks:
            x = blk(x)
        x = x.flatten(3).mean(3)
        x = self.head(self.head_lif(x))
        return x.mean(0)


# === Smoke test all 3 modes ===
for mode in ["SDT", "STAtten", "MSB_STAtten"]:
    m = SpikingTransformer(32, 3, 10, 256, 8, 2, 4, 4, 2, "0011", mode).cuda()
    o = m(torch.randn(2, 3, 32, 32).cuda())
    p = sum(x.numel() for x in m.parameters()) / 1e6
    print(f"{mode:<15s} output={o.shape} params={p:.2f}M")
    functional.reset_net(m); del m
torch.cuda.empty_cache()
print("All 3 modes work.")

SDT             output=torch.Size([2, 10]) params=2.57M
STAtten         output=torch.Size([2, 10]) params=2.57M
MSB_STAtten     output=torch.Size([2, 10]) params=2.57M
All 3 modes work.


In [3]:
#@title 3. Data Loading

import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader

def get_loaders(dataset='cifar10', bs=64, nw=4):
    if dataset == 'cifar10':
        DS = torchvision.datasets.CIFAR10
        mean, std, nc = [0.4914,0.4822,0.4465], [0.2470,0.2435,0.2616], 10
    else:
        DS = torchvision.datasets.CIFAR100
        mean, std, nc = [0.5071,0.4867,0.4408], [0.2675,0.2565,0.2761], 100
    tr = T.Compose([T.RandomCrop(32,4), T.RandomHorizontalFlip(),
                    T.AutoAugment(T.AutoAugmentPolicy.CIFAR10),
                    T.ToTensor(), T.Normalize(mean, std)])
    te = T.Compose([T.ToTensor(), T.Normalize(mean, std)])
    trl = DataLoader(DS('./data',True,download=True,transform=tr), bs, True, num_workers=nw, pin_memory=True, drop_last=True)
    tel = DataLoader(DS('./data',False,download=True,transform=te), bs, False, num_workers=nw, pin_memory=True)
    return trl, tel, nc

trl, tel, nc = get_loaders('cifar10', 4)
print(f"Test batch: {next(iter(trl))[0].shape} | classes={nc}")
del trl, tel
print("Data OK.")

Test batch: torch.Size([4, 3, 32, 32]) | classes=10
Data OK.


In [4]:
#@title 4. Training Engine

import time, os

def train_epoch(model, loader, opt, dev):
    model.train()
    loss_sum, ok, n = 0, 0, 0
    crit = nn.CrossEntropyLoss(label_smoothing=0.1)
    for imgs, tgts in loader:
        imgs, tgts = imgs.to(dev), tgts.to(dev)
        lam = torch.distributions.Beta(0.5, 0.5).sample().item()
        idx = torch.randperm(imgs.size(0), device=dev)
        imgs = lam * imgs + (1-lam) * imgs[idx]
        out = model(imgs)
        loss = lam * crit(out, tgts) + (1-lam) * crit(out, tgts[idx])
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); functional.reset_net(model)
        loss_sum += loss.item(); ok += out.argmax(1).eq(tgts).sum().item(); n += tgts.size(0)
    return loss_sum/len(loader), 100.*ok/n

@torch.no_grad()
def eval_model(model, loader, dev):
    model.eval()
    ok, n = 0, 0
    for imgs, tgts in loader:
        imgs, tgts = imgs.to(dev), tgts.to(dev)
        out = model(imgs)
        functional.reset_net(model)
        ok += out.argmax(1).eq(tgts).sum().item(); n += tgts.size(0)
    return 100.*ok/n

def run_experiment(attn_mode, dataset='cifar10', dims=256, heads=8, depth=2,
                   T_steps=2, epochs=100, bs=64, lr=3e-4):
    dev = torch.device('cuda')
    trl, tel, nc = get_loaders(dataset, bs)
    model = SpikingTransformer(32, 3, nc, dims, heads, depth, 4, T_steps, 2, "0011", attn_mode).to(dev)
    npar = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"\n{'='*60}\n{attn_mode} on {dataset.upper()} | {npar:.2f}M params | T={T_steps} | {epochs} ep\n{'='*60}")

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.06)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-5)
    warmup = 10
    best, hist = 0, []

    for ep in range(epochs):
        if ep < warmup:
            for pg in opt.param_groups: pg['lr'] = lr*(ep+1)/warmup
        t0 = time.time()
        tl, ta = train_epoch(model, trl, opt, dev)
        va = eval_model(model, tel, dev)
        sched.step()
        hist.append(va)
        if va > best:
            best = va
            os.makedirs('ckpt', exist_ok=True)
            torch.save(model.state_dict(), f'ckpt/{attn_mode}_{dataset}.pt')
        if ep % 10 == 0 or ep == epochs-1:
            print(f"Ep {ep:3d}/{epochs} | train {ta:.2f}% loss {tl:.4f} | test {va:.2f}% | best {best:.2f}% | {time.time()-t0:.1f}s")

    print(f"\nBest: {best:.2f}%")
    del model, opt, sched, trl, tel; torch.cuda.empty_cache()
    return best, hist

print("Training engine ready.")

Training engine ready.


---
## 5. Run Experiments (SDT-2-256, T=4, 100 epochs, bs=32)

In [5]:
acc_c10_sdt, hist_c10_sdt = run_experiment('SDT', 'cifar10')


SDT on CIFAR10 | 2.57M params | T=2 | 100 ep
Ep   0/100 | train 15.85% loss 2.2021 | test 31.42% | best 31.42% | 76.8s
Ep  10/100 | train 32.47% loss 1.7191 | test 67.23% | best 67.23% | 78.2s
Ep  20/100 | train 38.75% loss 1.5568 | test 74.20% | best 75.80% | 78.1s
Ep  30/100 | train 40.55% loss 1.5086 | test 79.69% | best 79.69% | 78.0s
Ep  40/100 | train 41.58% loss 1.4743 | test 81.28% | best 82.11% | 78.1s
Ep  50/100 | train 41.52% loss 1.4227 | test 84.00% | best 84.00% | 78.1s
Ep  60/100 | train 43.86% loss 1.3809 | test 84.52% | best 85.31% | 78.0s
Ep  70/100 | train 45.00% loss 1.3646 | test 86.05% | best 86.14% | 77.9s
Ep  80/100 | train 44.12% loss 1.3606 | test 85.96% | best 86.62% | 78.1s
Ep  90/100 | train 44.09% loss 1.3531 | test 86.90% | best 86.90% | 78.0s
Ep  99/100 | train 44.74% loss 1.3328 | test 86.84% | best 87.12% | 78.0s

Best: 87.12%


In [6]:
acc_c10_sta, hist_c10_sta = run_experiment('STAtten', 'cifar10')


STAtten on CIFAR10 | 2.57M params | T=2 | 100 ep
Ep   0/100 | train 15.75% loss 2.1998 | test 31.92% | best 31.92% | 79.2s
Ep  10/100 | train 33.32% loss 1.6917 | test 67.30% | best 67.30% | 79.4s
Ep  20/100 | train 38.97% loss 1.5435 | test 77.90% | best 77.90% | 79.2s
Ep  30/100 | train 40.16% loss 1.4545 | test 81.55% | best 81.75% | 79.3s
Ep  40/100 | train 42.80% loss 1.4239 | test 84.72% | best 84.72% | 79.3s
Ep  50/100 | train 43.48% loss 1.3941 | test 85.32% | best 85.54% | 79.2s
Ep  60/100 | train 44.03% loss 1.3722 | test 86.93% | best 86.93% | 79.3s
Ep  70/100 | train 44.76% loss 1.3416 | test 87.86% | best 87.86% | 79.3s
Ep  80/100 | train 46.55% loss 1.3313 | test 87.90% | best 88.20% | 79.4s
Ep  90/100 | train 47.10% loss 1.3052 | test 88.38% | best 88.72% | 79.3s
Ep  99/100 | train 46.24% loss 1.3275 | test 88.72% | best 88.72% | 79.3s

Best: 88.72%


In [7]:
acc_c10_msb, hist_c10_msb = run_experiment('MSB_STAtten', 'cifar10')


MSB_STAtten on CIFAR10 | 2.57M params | T=2 | 100 ep
Ep   0/100 | train 16.11% loss 2.2008 | test 32.01% | best 32.01% | 80.9s
Ep  10/100 | train 35.13% loss 1.6840 | test 68.91% | best 68.91% | 81.2s
Ep  20/100 | train 39.70% loss 1.5080 | test 77.69% | best 78.00% | 80.9s
Ep  30/100 | train 41.49% loss 1.4558 | test 82.66% | best 82.66% | 81.0s
Ep  40/100 | train 42.90% loss 1.4280 | test 83.29% | best 84.49% | 81.0s
Ep  50/100 | train 44.61% loss 1.3771 | test 85.35% | best 85.95% | 81.0s
Ep  60/100 | train 45.51% loss 1.3600 | test 86.43% | best 86.76% | 81.0s
Ep  70/100 | train 44.33% loss 1.3367 | test 87.41% | best 88.03% | 81.0s
Ep  80/100 | train 43.40% loss 1.3294 | test 87.93% | best 88.15% | 80.9s
Ep  90/100 | train 44.50% loss 1.3223 | test 88.31% | best 88.86% | 80.9s
Ep  99/100 | train 44.74% loss 1.2923 | test 88.62% | best 88.91% | 80.9s

Best: 88.91%


In [8]:
acc_c100_sdt, hist_c100_sdt = run_experiment('SDT', 'cifar100')

100%|██████████| 169M/169M [00:04<00:00, 35.0MB/s] 



SDT on CIFAR100 | 2.59M params | T=2 | 100 ep
Ep   0/100 | train 2.17% loss 4.5349 | test 5.86% | best 5.86% | 78.2s
Ep  10/100 | train 9.54% loss 3.8954 | test 24.15% | best 24.15% | 78.6s
Ep  20/100 | train 16.00% loss 3.5517 | test 40.19% | best 40.19% | 78.6s
Ep  30/100 | train 19.14% loss 3.2965 | test 47.73% | best 48.27% | 78.6s
Ep  40/100 | train 22.89% loss 3.1991 | test 53.30% | best 53.30% | 78.5s
Ep  50/100 | train 21.70% loss 3.1366 | test 53.90% | best 55.70% | 78.7s
Ep  60/100 | train 25.67% loss 3.0306 | test 57.84% | best 58.00% | 78.6s
Ep  70/100 | train 25.36% loss 2.9919 | test 59.74% | best 59.92% | 78.6s
Ep  80/100 | train 26.95% loss 2.9663 | test 60.77% | best 60.90% | 78.5s
Ep  90/100 | train 27.87% loss 2.9011 | test 60.34% | best 61.32% | 78.6s
Ep  99/100 | train 26.94% loss 2.9124 | test 60.34% | best 61.92% | 78.5s

Best: 61.92%


In [9]:
acc_c100_sta, hist_c100_sta = run_experiment('STAtten', 'cifar100')


STAtten on CIFAR100 | 2.59M params | T=2 | 100 ep
Ep   0/100 | train 2.32% loss 4.5349 | test 6.27% | best 6.27% | 79.2s
Ep  10/100 | train 10.72% loss 3.8266 | test 26.71% | best 26.71% | 79.4s
Ep  20/100 | train 17.49% loss 3.3897 | test 48.06% | best 48.06% | 79.6s
Ep  30/100 | train 21.84% loss 3.1355 | test 53.67% | best 53.67% | 79.6s
Ep  40/100 | train 25.71% loss 3.0010 | test 58.55% | best 58.55% | 79.5s
Ep  50/100 | train 25.05% loss 2.9600 | test 60.78% | best 60.78% | 79.6s
Ep  60/100 | train 26.32% loss 2.8512 | test 62.53% | best 62.82% | 79.6s
Ep  70/100 | train 29.31% loss 2.7812 | test 64.58% | best 64.58% | 79.8s
Ep  80/100 | train 29.73% loss 2.7274 | test 65.30% | best 65.59% | 79.7s
Ep  90/100 | train 27.58% loss 2.7816 | test 66.36% | best 66.36% | 79.7s
Ep  99/100 | train 30.56% loss 2.7544 | test 66.25% | best 66.70% | 79.6s

Best: 66.70%


In [ ]:
acc_c100_msb, hist_c100_msb = run_experiment('MSB_STAtten', 'cifar100')

In [ ]:
#@title 6. Results Summary

print("=" * 80)
print(f"{'Method':<22} {'CIFAR-10':<12} {'(paper)':<12} {'CIFAR-100':<12} {'(paper)'}")
print("=" * 80)
for name, c10, c10r, c100, c100r in [
    ('SDT (spatial-only)',  acc_c10_sdt,  95.60, acc_c100_sdt,  78.40),
    ('STAtten (B=2)',      acc_c10_sta,  96.03, acc_c100_sta,  79.85),
    ('MSB-STAtten (ours)', acc_c10_msb,  None,  acc_c100_msb,  None),
]:
    r10 = f"{c10r:.2f}%" if c10r else "-"
    r100 = f"{c100r:.2f}%" if c100r else "-"
    tag = " ***" if 'MSB' in name else ""
    print(f"{name:<22} {c10:.2f}%{'':<6} {r10:<12} {c100:.2f}%{'':<6} {r100}{tag}")
print("=" * 80)

In [ ]:
#@title 7. Plots

import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
colors = ['#95A5A6', '#F39C12', '#2980B9']
methods = ['SDT', 'STAtten', 'MSB-STAtten']

# Bar charts
for ax, ds, ours, paper, yl in [
    (axes[0,0], 'CIFAR-10',  [acc_c10_sdt, acc_c10_sta, acc_c10_msb],   [95.60,96.03,None], (93,97)),
    (axes[0,1], 'CIFAR-100', [acc_c100_sdt, acc_c100_sta, acc_c100_msb], [78.40,79.85,None], (75,82)),
]:
    x = np.arange(3); w = 0.35
    pv = [v if v else 0 for v in paper]
    ax.bar(x-w/2, pv, w, label='Paper', color=colors, alpha=0.4, edgecolor='k', lw=0.5)
    ax.bar(x+w/2, ours, w, label='Ours', color=colors, edgecolor='k', lw=0.5)
    for i,(p,o) in enumerate(zip(paper, ours)):
        if p: ax.text(i-w/2, p+0.05, f'{p:.1f}', ha='center', fontsize=8)
        ax.text(i+w/2, o+0.05, f'{o:.2f}', ha='center', fontsize=8, fontweight='bold')
    ax.set_xticks(x); ax.set_xticklabels(methods); ax.set_ylim(yl)
    ax.set_title(ds, fontweight='bold'); ax.legend(); ax.grid(axis='y', alpha=0.3)

# Curves
for ax, ds, hists in [
    (axes[1,0], 'CIFAR-10',  [hist_c10_sdt, hist_c10_sta, hist_c10_msb]),
    (axes[1,1], 'CIFAR-100', [hist_c100_sdt, hist_c100_sta, hist_c100_msb]),
]:
    for h, l, c in zip(hists, methods, colors):
        ax.plot(h, label=l, color=c, lw=1.5)
    ax.set_xlabel('Epoch'); ax.set_ylabel('Test Acc (%)')
    ax.set_title(ds+' — Training', fontweight='bold'); ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('results.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: results.png")